# Pydantic Homework

## Homework 1: User Profile Model

Create a Pydantic `User` model with:

- `name`
- `age`
- `email`
- Optional `phone`
- `password` using `SecretStr`

Create an instance and practice **direct attribute access**.

Also serialize the model and deserialize it back into a Pydantic object.

In [3]:
from pydantic import BaseModel, SecretStr, EmailStr
from typing import Optional


class User(BaseModel):
    name: str
    age: int
    email: EmailStr
    phone: Optional[str] = None
    password: SecretStr


# Create an instance
user = User(
    name="John Doe",
    age=28,
    email="john.doe@example.com",
    phone="+1-555-1234",
    password="MyPassword123"
)

print("Name:", user.name)
print("Age:", user.age)
print("Email:", user.email)
print("Phone:", user.phone)

print("Password (masked):", user.password)
print("Actual Password:", user.password.get_secret_value())

user_dict = user.model_dump()
print("\nSerialized to dict:")
print(user_dict)

# Serialize to JSON
user_json = user.model_dump_json()
print("\nSerialized to JSON:")
print(user_json)

# Deserialize back to a Pydantic object
user_from_dict = User.model_validate(user_dict)

print("\nDeserialized User:")
print(user_from_dict)

print("\nType:", type(user_from_dict))

Name: John Doe
Age: 28
Email: john.doe@example.com
Phone: +1-555-1234
Password (masked): **********
Actual Password: MyPassword123

Serialized to dict:
{'name': 'John Doe', 'age': 28, 'email': 'john.doe@example.com', 'phone': '+1-555-1234', 'password': SecretStr('**********')}

Serialized to JSON:
{"name":"John Doe","age":28,"email":"john.doe@example.com","phone":"+1-555-1234","password":"**********"}

Deserialized User:
name='John Doe' age=28 email='john.doe@example.com' phone='+1-555-1234' password=SecretStr('**********')

Type: <class '__main__.User'>


## Homework 2: Nested Order Model

Create Pydantic models for:

- `Customer`
- `Address`
- `Product`
- `Order`

An `Order` should contain a nested `Customer`, an `Address`, and a collection of `Product` objects.

Validate the input and access nested fields using normal attribute access.

In [5]:
from pydantic import BaseModel, EmailStr
from typing import List


class Address(BaseModel):
    street: str
    city: str
    state: str
    zip_code: str


class Customer(BaseModel):
    name: str
    email: EmailStr


class Product(BaseModel):
    id: int
    name: str
    price: float
    quantity: int


class Order(BaseModel):
    order_id: int
    customer: Customer
    shipping_address: Address
    products: List[Product]


# Input data
order_data = {
    "order_id": 1001,
    "customer": {
        "name": "John Doe",
        "email": "john.doe@example.com"
    },
    "shipping_address": {
        "street": "123 Main St",
        "city": "New York",
        "state": "NY",
        "zip_code": "10001"
    },
    "products": [
        {
            "id": 1,
            "name": "Laptop",
            "price": 1200.50,
            "quantity": 1
        },
        {
            "id": 2,
            "name": "Mouse",
            "price": 25.99,
            "quantity": 2
        }
    ]
}

# Validation happens automatically here
order = Order.model_validate(order_data)

print("Order validated successfully!\n")

print("Customer Name:", order.customer.name)
print("Customer Email:", order.customer.email)

print("\nShipping City:", order.shipping_address.city)
print("Shipping State:", order.shipping_address.state)

print("\nFirst Product:", order.products[0].name)
print("First Product Price:", order.products[0].price)

total = sum(product.price * product.quantity for product in order.products)
print("\nTotal Order Value:", total)

Order validated successfully!

Customer Name: John Doe
Customer Email: john.doe@example.com

Shipping City: New York
Shipping State: NY

First Product: Laptop
First Product Price: 1200.5

Total Order Value: 1252.48


## Homework 3: Validated API Request

Create a `UserRequest` model with:

- Required and optional fields
- An aliased field using `Field`
- Constraints such as minimum/maximum length and numeric limits
- `Annotated` for at least one field

Add a custom validator for one business rule.

In [8]:
from typing import Optional, Annotated
from pydantic import BaseModel, Field, field_validator

class UserRequest(BaseModel):
    username: Annotated[
        str,
        Field(min_length=3, max_length=20)
    ]

    full_name: str = Field(alias="fullName")

    age: int = Field(ge=18, le=100)

    email: Annotated[
        str,
        Field(min_length=5, max_length=100)
    ]

    referral_code: Optional[str] = None

    @field_validator("username")
    @classmethod
    def username_must_not_contain_admin(cls, value):
        """
        Business Rule:
        Username cannot contain the word 'admin'.
        """
        if "admin" in value.lower():
            raise ValueError("Username cannot contain 'admin'")
        return value

request_data = {
    "username": "john_doe",
    "fullName": "John Doe",
    "age": 25,
    "email": "john@example.com",
    "referral_code": "WELCOME2024"
}

user_request = UserRequest.model_validate(request_data)

print("Validated Request:")
print(user_request)

print("\nUsername:", user_request.username)
print("Full Name:", user_request.full_name)
print("Age:", user_request.age)
print("Email:", user_request.email)
print("Referral Code:", user_request.referral_code)

Validated Request:
username='john_doe' full_name='John Doe' age=25 email='john@example.com' referral_code='WELCOME2024'

Username: john_doe
Full Name: John Doe
Age: 25
Email: john@example.com
Referral Code: WELCOME2024


## Homework 4: Application Configuration

Create a Pydantic model representing application configuration.

Include:

- API key using `SecretStr`
- Environment/name fields
- A dictionary of model configuration
- Appropriate field constraints
- At least one optional field

Serialize the configuration and verify that sensitive values are handled appropriately.

In [11]:
from typing import Optional, Dict, Any
from pydantic import BaseModel, Field, SecretStr

class AppConfig(BaseModel):
    app_name: str = Field(
        min_length=3,
        max_length=50,
        description="Application name"
    )

    environment: str = Field(
        pattern="^(dev|test|uat|prod)$",
        description="Deployment environment"
    )

    api_key: SecretStr

    model_settings: Dict[str, Any]

    timeout_seconds: int = Field(
        default=30,
        ge=1,
        le=300
    )

    log_level: Optional[str] = Field(
        default=None,
        pattern="^(DEBUG|INFO|WARNING|ERROR|CRITICAL)$"
    )

config = AppConfig(
    app_name="CustomerPortal",
    environment="prod",
    api_key="super-secret-api-key",
    model_settings={
        "model_name": "gpt-4",
        "temperature": 0.7,
        "max_tokens": 1000
    },
    timeout_seconds=60,
    log_level="INFO"
)

print("Configuration:")
print(config)

print("\nApp Name:", config.app_name)
print("Environment:", config.environment)
print("Model Name:", config.model_settings["model_name"])

print("\nMasked API Key:", config.api_key)
print("Actual API Key:", config.api_key.get_secret_value())

config_dict = config.model_dump()
print("\nSerialized Dictionary:")
print(config_dict)

config_json = config.model_dump_json()
print("\nSerialized JSON:")
print(config_json)

restored_config = AppConfig.model_validate(config_dict)

print("\nRestored Configuration:")
print(restored_config)

Configuration:
app_name='CustomerPortal' environment='prod' api_key=SecretStr('**********') model_settings={'model_name': 'gpt-4', 'temperature': 0.7, 'max_tokens': 1000} timeout_seconds=60 log_level='INFO'

App Name: CustomerPortal
Environment: prod
Model Name: gpt-4

Masked API Key: **********
Actual API Key: super-secret-api-key

Serialized Dictionary:
{'app_name': 'CustomerPortal', 'environment': 'prod', 'api_key': SecretStr('**********'), 'model_settings': {'model_name': 'gpt-4', 'temperature': 0.7, 'max_tokens': 1000}, 'timeout_seconds': 60, 'log_level': 'INFO'}

Serialized JSON:
{"app_name":"CustomerPortal","environment":"prod","api_key":"**********","model_settings":{"model_name":"gpt-4","temperature":0.7,"max_tokens":1000},"timeout_seconds":60,"log_level":"INFO"}

Restored Configuration:
app_name='CustomerPortal' environment='prod' api_key=SecretStr('**********') model_settings={'model_name': 'gpt-4', 'temperature': 0.7, 'max_tokens': 1000} timeout_seconds=60 log_level='INFO'


## Homework 5: LLM Response Pipeline

Create a Pydantic model representing a structured LLM response:

- `answer`
- `confidence`
- `sources`
- `metadata`

Add appropriate validation constraints and one custom validation rule.

Simulate the complete pipeline:

```text
JSON Response
     ↓
Pydantic Validation
     ↓
Python Object
     ↓
Application Logic
     ↓
Serialization
     ↓
JSON

In [13]:
from typing import List, Dict, Any
from pydantic import BaseModel, Field, field_validator

class LLMResponse(BaseModel):
    answer: str = Field(
        min_length=10,
        max_length=1000,
        description="Generated answer"
    )

    confidence: float = Field(
        ge=0.0,
        le=1.0,
        description="Confidence score between 0 and 1"
    )

    sources: List[str] = Field(
        min_length=1,
        description="List of references or sources"
    )

    metadata: Dict[str, Any]

    @field_validator("sources")
    @classmethod
    def sources_cannot_be_empty_strings(cls, value):
        """
        Business Rule:
        Every source must contain meaningful text.
        """
        for source in value:
            if not source.strip():
                raise ValueError("Sources cannot contain empty values")
        return value

# Step 1: JSON Response from an LLM

json_response = {
    "answer": "Pydantic validates structured data and converts it into Python objects.",
    "confidence": 0.92,
    "sources": [
        "https://docs.pydantic.dev",
        "https://fastapi.tiangolo.com"
    ],
    "metadata": {
        "model": "gpt-4",
        "tokens_used": 145,
        "response_time_ms": 230
    }
}

print("STEP 1: Raw JSON Response")
print(json_response)

# Step 2: Pydantic Validation

validated_response = LLMResponse.model_validate(json_response)

print("\nSTEP 2: Validation Successful")

# Step 3: Python Object

print("\nSTEP 3: Python Object")
print(validated_response)

print("\nAnswer:")
print(validated_response.answer)

print("\nConfidence:")
print(validated_response.confidence)

# Step 4: Application Logic

print("\nSTEP 4: Application Logic")

if validated_response.confidence >= 0.80:
    status = "High Confidence"
else:
    status = "Needs Review"

print("Status:", status)

# Step 5: Serialization

serialized_data = validated_response.model_dump()

print("\nSTEP 5: Serialized Dictionary")
print(serialized_data)

# Step 6: Convert Back to JSON

json_output = validated_response.model_dump_json(indent=2)

print("\nSTEP 6: Final JSON Output")
print(json_output)

STEP 1: Raw JSON Response
{'answer': 'Pydantic validates structured data and converts it into Python objects.', 'confidence': 0.92, 'sources': ['https://docs.pydantic.dev', 'https://fastapi.tiangolo.com'], 'metadata': {'model': 'gpt-4', 'tokens_used': 145, 'response_time_ms': 230}}

STEP 2: Validation Successful

STEP 3: Python Object
answer='Pydantic validates structured data and converts it into Python objects.' confidence=0.92 sources=['https://docs.pydantic.dev', 'https://fastapi.tiangolo.com'] metadata={'model': 'gpt-4', 'tokens_used': 145, 'response_time_ms': 230}

Answer:
Pydantic validates structured data and converts it into Python objects.

Confidence:
0.92

STEP 4: Application Logic
Status: High Confidence

STEP 5: Serialized Dictionary
{'answer': 'Pydantic validates structured data and converts it into Python objects.', 'confidence': 0.92, 'sources': ['https://docs.pydantic.dev', 'https://fastapi.tiangolo.com'], 'metadata': {'model': 'gpt-4', 'tokens_used': 145, 'response_t